In [ ]:
# Create model
print(f"Creating model: {config.model_name}")
model = create_model(
    config.model_name,
    num_classes=config.num_classes,
    drop_rate=0.1,
    drop_path_rate=0.1
)
model = model.to(config.device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Loss function with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

# Learning rate scheduler
num_steps = len(train_loader) * config.num_epochs
warmup_steps = len(train_loader) * config.warmup_epochs

if config.lr_scheduler == 'cosine':
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, num_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))
    
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
else:
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# Mixed precision training
scaler = GradScaler() if config.use_amp else None

# TensorBoard writer
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter(log_dir / f'{config.model_name}_{timestamp}')

print("\nTraining setup complete!")

## 7. Evaluation and Results

In [ ]:
# Test evaluation
print("\nEvaluating on test set...")
test_loss, test_acc, test_preds, test_targets, test_probs = validate(
    model, test_loader, criterion, config, -1, writer
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

# Detailed classification report
from sklearn.metrics import classification_report

class_names = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R', 'Analysis']
print("\nClassification Report:")
print(classification_report(test_targets, test_preds, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(test_targets, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Plot training curves
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Val Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
# Per-class accuracy
class_accuracies = []
for i in range(len(class_names)):
    class_mask = np.array(test_targets) == i
    if class_mask.sum() > 0:
        class_acc = np.mean(np.array(test_preds)[class_mask] == i) * 100
        class_accuracies.append(class_acc)
    else:
        class_accuracies.append(0)

plt.bar(class_names, class_accuracies)
plt.title('Per-Class Test Accuracy')
plt.xlabel('Class')
plt.ylabel('Accuracy (%)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Save final model and results
final_results = {
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'test_loss': test_loss,
    'train_losses': train_losses,
    'val_losses': val_losses,
    'train_accs': train_accs,
    'val_accs': val_accs,
    'class_accuracies': class_accuracies,
    'confusion_matrix': cm.tolist(),
    'config': config.to_dict()
}

# Save results
results_path = log_dir / f'{config.model_name}_results.json'
with open(results_path, 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"\nResults saved to: {results_path}")
print(f"Model checkpoints saved to: {checkpoint_dir}")
print(f"TensorBoard logs saved to: {writer.log_dir}")

# Close writer
writer.close()

## 6. Main Training Loop

In [ ]:
# Training loop
print("Starting training...")
start_time = time.time()

# Initialize tracking variables
best_val_acc = 0.0
train_losses = []
val_losses = []
train_accs = []
val_accs = []

# Training loop
for epoch in range(config.num_epochs):
    print(f"\nEpoch {epoch+1}/{config.num_epochs}")
    print("-" * 50)
    
    # Train for one epoch
    train_loss, train_acc, train_preds, train_targets = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, scaler, config, epoch, writer
    )
    
    # Validate
    val_loss, val_acc, val_preds, val_targets, val_probs = validate(
        model, val_loader, criterion, config, epoch, writer
    )
    
    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    # Print epoch summary
    print(f"\nTrain Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    # Save checkpoint
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
    
    if (epoch + 1) % config.save_interval == 0 or is_best:
        save_checkpoint(model, optimizer, scheduler, epoch, val_acc, 
                       checkpoint_dir, config, is_best)
    
    # Early stopping (optional)
    if val_acc > 99.0:  # Stop if we reach very high accuracy
        print(f"Early stopping at epoch {epoch+1} with validation accuracy {val_acc:.2f}%")
        break

# Training completed
total_time = time.time() - start_time
print(f"\nTraining completed in {total_time:.2f} seconds")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

## 5. Training Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, config, epoch, writer):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Metrics tracking
    all_predictions = []
    all_targets = []
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.num_epochs}')
    
    for batch_idx, (data, target) in enumerate(pbar):
        data, target = data.to(config.device), target.to(config.device)
        
        # Apply mixup/cutmix augmentation
        if np.random.rand() < config.mixup_prob:
            if np.random.rand() < 0.5:  # Mixup
                mixed_data, target_a, target_b, lam = mixup_data(data, target, config.mixup_alpha)
                
                if config.use_amp:
                    with autocast():
                        outputs = model(mixed_data)
                        loss = lam * criterion(outputs, target_a) + (1 - lam) * criterion(outputs, target_b)
                else:
                    outputs = model(mixed_data)
                    loss = lam * criterion(outputs, target_a) + (1 - lam) * criterion(outputs, target_b)
            else:  # CutMix
                mixed_data, target_a, target_b, lam = cutmix_data(data, target, config.cutmix_alpha)
                
                if config.use_amp:
                    with autocast():
                        outputs = model(mixed_data)
                        loss = lam * criterion(outputs, target_a) + (1 - lam) * criterion(outputs, target_b)
                else:
                    outputs = model(mixed_data)
                    loss = lam * criterion(outputs, target_a) + (1 - lam) * criterion(outputs, target_b)
        else:
            # Standard training
            if config.use_amp:
                with autocast():
                    outputs = model(data)
                    loss = criterion(outputs, target)
            else:
                outputs = model(data)
                loss = criterion(outputs, target)
        
        # Backward pass
        optimizer.zero_grad()
        
        if config.use_amp and scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
            optimizer.step()
        
        # Update learning rate
        scheduler.step()
        
        # Update metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += target.size(0)
        
        if 'mixup_prob' in locals() and np.random.rand() < config.mixup_prob:
            # For mixup/cutmix, use soft accuracy
            correct += (lam * predicted.eq(target_a).sum().item() + 
                       (1 - lam) * predicted.eq(target_b).sum().item())
        else:
            correct += predicted.eq(target).sum().item()
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
        
        # Update progress bar
        accuracy = 100. * correct / total
        pbar.set_postfix({
            'Loss': f'{running_loss/(batch_idx+1):.4f}',
            'Acc': f'{accuracy:.2f}%',
            'LR': f'{scheduler.get_last_lr()[0]:.2e}'
        })
        
        # Log to tensorboard
        if batch_idx % config.log_interval == 0:
            step = epoch * len(train_loader) + batch_idx
            writer.add_scalar('Train/Loss', loss.item(), step)
            writer.add_scalar('Train/Accuracy', accuracy, step)
            writer.add_scalar('Train/LearningRate', scheduler.get_last_lr()[0], step)
    
    # Calculate final metrics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc, all_predictions, all_targets


def validate(model, val_loader, criterion, config, epoch, writer):
    """Validate the model."""
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    all_predictions = []
    all_targets = []
    all_probs = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        
        for data, target in pbar:
            data, target = data.to(config.device), target.to(config.device)
            
            if config.use_amp:
                with autocast():
                    outputs = model(data)
                    loss = criterion(outputs, target)
            else:
                outputs = model(data)
                loss = criterion(outputs, target)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
            
            # Store predictions and probabilities
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
            
            # Update progress bar
            accuracy = 100. * correct / total
            pbar.set_postfix({
                'Loss': f'{val_loss/(len(pbar)):.4f}',
                'Acc': f'{accuracy:.2f}%'
            })
    
    # Calculate metrics
    val_loss /= len(val_loader)
    val_acc = 100. * correct / total
    
    # Calculate detailed metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets, all_predictions, average='weighted', zero_division=0
    )
    
    # Log to tensorboard
    writer.add_scalar('Val/Loss', val_loss, epoch)
    writer.add_scalar('Val/Accuracy', val_acc, epoch)
    writer.add_scalar('Val/Precision', precision, epoch)
    writer.add_scalar('Val/Recall', recall, epoch)
    writer.add_scalar('Val/F1', f1, epoch)
    
    return val_loss, val_acc, all_predictions, all_targets, all_probs


def save_checkpoint(model, optimizer, scheduler, epoch, val_acc, checkpoint_dir, config, is_best=False):
    """Save model checkpoint."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_acc': val_acc,
        'config': config.to_dict()
    }
    
    # Save regular checkpoint
    checkpoint_path = checkpoint_dir / f'{config.model_name}_epoch_{epoch}.pth'
    torch.save(checkpoint, checkpoint_path)
    
    # Save best model
    if is_best:
        best_path = checkpoint_dir / f'{config.model_name}_best.pth'
        torch.save(checkpoint, best_path)
        print(f"New best model saved with validation accuracy: {val_acc:.2f}%")
    
    return checkpoint_path

## 4. Model and Training Setup

In [ ]:
def mixup_data(x, y, alpha=1.0):
    """Mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


def cutmix_data(x, y, alpha=1.0):
    """CutMix augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    
    # Adjust lambda to exactly match pixel ratio
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam


def rand_bbox(size, lam):
    """Generate random bounding box for CutMix."""
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = np.int32(W * cut_rat)
    cut_h = np.int32(H * cut_rat)

    # Uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2


# Create data loaders
print("Creating data loaders...")

# Create datasets
train_dataset = PacketImageDataset(
    train_df, byte_columns, 
    image_size=(config.image_size, config.image_size),
    encoding_method=config.encoding_method,
    use_torch_encoder=False  # Use numpy encoder for flexibility
)

val_dataset = PacketImageDataset(
    val_df, byte_columns,
    image_size=(config.image_size, config.image_size),
    encoding_method=config.encoding_method,
    use_torch_encoder=False
)

test_dataset = PacketImageDataset(
    test_df, byte_columns,
    image_size=(config.image_size, config.image_size),
    encoding_method=config.encoding_method,
    use_torch_encoder=False
)

# Create loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 3. Data Augmentation and Loaders

In [ ]:
class TrainingConfig:
    """Training configuration."""
    
    # Model settings
    model_name: str = 'vit_packet_small'
    image_size: int = 128
    num_classes: int = 6
    
    # Training settings
    batch_size: int = 32
    num_epochs: int = 50
    learning_rate: float = 3e-4
    weight_decay: float = 0.05
    warmup_epochs: int = 5
    
    # Optimization
    use_amp: bool = True  # Automatic Mixed Precision
    gradient_clip: float = 1.0
    
    # Learning rate schedule
    lr_scheduler: str = 'cosine'  # 'cosine' or 'step'
    lr_min: float = 1e-6
    
    # Data settings
    encoding_method: str = 'sequential'
    num_workers: int = 4
    pin_memory: bool = True
    
    # Regularization
    label_smoothing: float = 0.1
    mixup_alpha: float = 0.2
    cutmix_alpha: float = 1.0
    mixup_prob: float = 0.5
    
    # Logging
    log_interval: int = 10
    save_interval: int = 5
    
    # Device
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    def to_dict(self):
        return {k: v for k, v in self.__class__.__dict__.items() 
                if not k.startswith('_') and not callable(v)}

config = TrainingConfig()
print("Training Configuration:")
for key, value in config.to_dict().items():
    print(f"  {key}: {value}")

## 2. Training Configuration

In [ ]:
# Load data
print("Loading packet data...")
cic_data = pd.read_csv(raw_data_dir / 'payload_byte' / 'cic_ids2017_sample.csv')
unsw_data = pd.read_csv(raw_data_dir / 'payload_byte' / 'unsw_nb15_sample.csv')

# Combine datasets
all_data = pd.concat([cic_data, unsw_data], ignore_index=True)
print(f"Total samples: {len(all_data)}")
print(f"Label distribution:\n{all_data['label'].value_counts().sort_index()}")

# Define byte columns
byte_columns = [f'byte_{i}' for i in range(1500)]

# Split data
X = all_data[byte_columns].values
y = all_data['label'].values

# Create train/val/test splits (70/15/15)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp  # 0.176 * 0.85 ≈ 0.15
)

# Create DataFrames for datasets
train_df = pd.DataFrame(X_train, columns=byte_columns)
train_df['label'] = y_train

val_df = pd.DataFrame(X_val, columns=byte_columns)
val_df['label'] = y_val

test_df = pd.DataFrame(X_test, columns=byte_columns)
test_df['label'] = y_test

print(f"\nDataset splits:")
print(f"Train: {len(train_df)} samples")
print(f"Val: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

### 1.2 Load and Prepare Data

In [ ]:
class PacketImageDataset(Dataset):
    """
    PyTorch Dataset for packet images.
    """
    
    def __init__(self, 
                 data: pd.DataFrame,
                 byte_columns: List[str],
                 label_column: str = 'label',
                 image_size: Tuple[int, int] = (128, 128),
                 encoding_method: str = 'sequential',
                 transform: Optional[Any] = None,
                 use_torch_encoder: bool = True):
        """
        Args:
            data: DataFrame containing packet data
            byte_columns: List of column names containing byte values
            label_column: Name of label column
            image_size: Size of output images
            encoding_method: Method for encoding bytes to images
            transform: Optional torchvision transforms
            use_torch_encoder: Whether to use GPU-accelerated encoder
        """
        self.data = data
        self.byte_columns = byte_columns
        self.label_column = label_column
        self.transform = transform
        
        # Extract features and labels
        self.packet_bytes = data[byte_columns].values
        self.labels = data[label_column].values
        
        # Initialize encoder
        if use_torch_encoder:
            self.encoder = TorchPacketEncoder(image_size, encoding_method)
        else:
            self.encoder = PacketImageEncoder(image_size)
            self.encoding_method = encoding_method
            self.use_torch_encoder = False
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get packet bytes
        packet = self.packet_bytes[idx]
        label = self.labels[idx]
        
        # Encode to image
        if hasattr(self.encoder, 'forward'):  # TorchPacketEncoder
            # Convert to tensor and add batch dimension
            packet_tensor = torch.tensor(packet, dtype=torch.float32).unsqueeze(0)
            image = self.encoder(packet_tensor).squeeze(0)  # Remove batch dim
        else:  # PacketImageEncoder
            if self.encoding_method == 'sequential':
                image_np = self.encoder.encode_sequential(packet)
            elif self.encoding_method == 'hilbert':
                image_np = self.encoder.encode_hilbert_curve(packet)
            else:
                image_np = self.encoder.encode_spiral(packet)
            
            # Convert to tensor and normalize
            image = torch.tensor(image_np, dtype=torch.float32).unsqueeze(0) / 255.0
        
        # Apply transforms if any
        if self.transform:
            image = self.transform(image)
            
        return image, label


class BalancedBatchSampler:
    """
    Sampler that ensures balanced classes in each batch.
    """
    
    def __init__(self, labels: np.ndarray, batch_size: int, num_classes: int):
        self.labels = labels
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.samples_per_class = batch_size // num_classes
        
        # Group indices by class
        self.class_indices = {}
        for class_id in range(num_classes):
            self.class_indices[class_id] = np.where(labels == class_id)[0]
            
    def __iter__(self):
        # Shuffle indices within each class
        for class_id in self.class_indices:
            np.random.shuffle(self.class_indices[class_id])
            
        # Generate balanced batches
        max_samples = min(len(indices) for indices in self.class_indices.values())
        num_batches = max_samples // self.samples_per_class
        
        for batch_idx in range(num_batches):
            batch = []
            for class_id in range(self.num_classes):
                start_idx = batch_idx * self.samples_per_class
                end_idx = start_idx + self.samples_per_class
                batch.extend(self.class_indices[class_id][start_idx:end_idx])
            
            # Shuffle within batch
            np.random.shuffle(batch)
            yield batch
            
    def __len__(self):
        max_samples = min(len(indices) for indices in self.class_indices.values())
        return max_samples // self.samples_per_class

## 1. Data Preparation

### 1.1 Custom Dataset Class

In [ ]:
# Setup project paths and imports
notebook_path = Path().resolve()
project_root = notebook_path.parent

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import our modules
from src.data.packet_to_image import PacketImageEncoder, TorchPacketEncoder
from src.models.vit_packet import create_model, list_models

# Define paths
data_dir = project_root / 'data'
raw_data_dir = data_dir / 'raw'
model_dir = project_root / 'models'
log_dir = project_root / 'logs'
checkpoint_dir = project_root / 'checkpoints'

# Create directories
for dir_path in [model_dir, log_dir, checkpoint_dir]:
    dir_path.mkdir(exist_ok=True)

print(f"Project root: {project_root}")
print(f"Available models: {list_models()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import json
import time
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.cuda.amp import autocast, GradScaler

# ML metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split

# Progress tracking
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')

# Set random seeds for reproducibility
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 07 - Supervised Training Pipeline for Packet-ViT

This notebook implements a complete training pipeline for Vision Transformers on network packet data, including data loading, training loops, evaluation metrics, and experiment tracking.